# Stratified dataset split

Rebuild train, validation, and test splits with equal class representation. The original exported dataset remains unchanged.

In [ ]:
from __future__ import annotations

import hashlib
import random
import shutil
from collections import Counter
from pathlib import Path

SOURCE_ROOT = Path("../datasets/shoe_brands").resolve()
OUTPUT_ROOT = Path("../datasets/shoe_brands_stratified").resolve()
CLASSES = ("adidas", "converse", "nike")
SOURCE_SPLITS = ("train", "val", "test")
SPLIT_COUNTS_PER_CLASS = {"train": 193, "val": 55, "test": 27}
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
RANDOM_SEED = 37

if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Source dataset not found at {SOURCE_ROOT}. Run ../dataset/load_shoe_dataset.ipynb first."
    )

In [ ]:
def sha256(file_path: Path) -> str:
    digest = hashlib.sha256()
    with file_path.open("rb") as source_file:
        for chunk in iter(lambda: source_file.read(64 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


files_by_class = {}
for class_name in CLASSES:
    class_files = [
        file_path
        for split_name in SOURCE_SPLITS
        for file_path in sorted((SOURCE_ROOT / split_name / class_name).glob("*"))
        if file_path.is_file() and file_path.suffix.lower() in SUPPORTED_EXTENSIONS
    ]
    files_by_class[class_name] = class_files

source_counts = {class_name: len(files) for class_name, files in files_by_class.items()}
expected_class_total = sum(SPLIT_COUNTS_PER_CLASS.values())
if any(image_count != expected_class_total for image_count in source_counts.values()):
    raise ValueError(
        f"Every class must contain {expected_class_total} images; found {source_counts}."
    )

all_source_files = [file_path for files in files_by_class.values() for file_path in files]
source_hashes = [sha256(file_path) for file_path in all_source_files]
duplicate_hashes = [digest for digest, count in Counter(source_hashes).items() if count > 1]
if duplicate_hashes:
    raise ValueError(f"Source dataset contains {len(duplicate_hashes)} exact duplicate groups.")

print(f"Source counts: {source_counts}")
print(f"Source images: {len(all_source_files)}")
print("Source exact duplicates: 0")

In [ ]:
split_assignments = []
for class_index, class_name in enumerate(CLASSES):
    shuffled_files = files_by_class[class_name].copy()
    random.Random(RANDOM_SEED + class_index).shuffle(shuffled_files)

    start_index = 0
    for split_name, split_count in SPLIT_COUNTS_PER_CLASS.items():
        split_files = shuffled_files[start_index:start_index + split_count]
        split_assignments.extend(
            (source_path, split_name, class_name, output_index)
            for output_index, source_path in enumerate(split_files)
        )
        start_index += split_count

if len(split_assignments) != len(all_source_files):
    raise RuntimeError("Not every source image received a split assignment.")

for source_path, split_name, class_name, output_index in split_assignments:
    output_directory = OUTPUT_ROOT / split_name / class_name
    output_directory.mkdir(parents=True, exist_ok=True)
    output_path = output_directory / f"{class_name}_{output_index:04d}{source_path.suffix.lower()}"
    shutil.copy2(source_path, output_path)

print(f"Stratified dataset written to: {OUTPUT_ROOT}")

In [ ]:
output_counts = {
    split_name: {
        class_name: len([
            path for path in (OUTPUT_ROOT / split_name / class_name).glob("*")
            if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
        ])
        for class_name in CLASSES
    }
    for split_name in SPLIT_COUNTS_PER_CLASS
}

for split_name, class_counts in output_counts.items():
    expected_count = SPLIT_COUNTS_PER_CLASS[split_name]
    if any(count != expected_count for count in class_counts.values()):
        raise RuntimeError(
            f"Unexpected counts for {split_name}: {class_counts}; expected {expected_count} per class."
        )

hash_splits = {}
for split_name in SPLIT_COUNTS_PER_CLASS:
    for class_name in CLASSES:
        for file_path in (OUTPUT_ROOT / split_name / class_name).glob("*"):
            if file_path.is_file() and file_path.suffix.lower() in SUPPORTED_EXTENSIONS:
                hash_splits.setdefault(sha256(file_path), set()).add(split_name)

cross_split_hashes = [digest for digest, splits in hash_splits.items() if len(splits) > 1]
if cross_split_hashes:
    raise RuntimeError(f"Found {len(cross_split_hashes)} exact duplicate groups across splits.")

print(f"Output counts: {output_counts}")
print(f"Total output images: {sum(sum(counts.values()) for counts in output_counts.values())}")
print("Cross-split exact duplicates: 0")

The stratified dataset is ready for `EDA.ipynb` and `../yolo11/setup_yolo11.ipynb`. No training is performed here.